In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import os

# ==========================================
# MILESTONE 6: Backend API Development
# ==========================================

# Initialize FastAPI app
app = FastAPI(
    title="Exam Anxiety Detector API",
    description="API for predicting exam anxiety levels from text using BERT.",
    version="1.0"
)

# Data model for incoming requests
class TextRequest(BaseModel):
    text: str

# Data model for responses
class PredictionResponse(BaseModel):
    anxiety_level: str
    confidence: float

# Global variables for model and tokenizer
model = None
tokenizer = None
MODEL_DIR = "./bert_anxiety_model"

# Map numeric predictions back to text labels
LABEL_MAP = {0: "Low Anxiety", 1: "Moderate Anxiety", 2: "High Anxiety"}

@app.on_event("startup")
async def load_model():
    """Loads the trained BERT model on API startup."""
    global model, tokenizer
    if os.path.exists(MODEL_DIR):
        try:
            tokenizer = BertTokenizer.from_pretrained(MODEL_DIR)
            model = BertForSequenceClassification.from_pretrained(MODEL_DIR)
            model.eval() # Set model to evaluation mode
            print("Successfully loaded the trained BERT model.")
        except Exception as e:
            print(f"Error loading model: {e}")
    else:
        print(f"WARNING: Model directory '{MODEL_DIR}' not found.")
        print("Please run train_model.py first to generate the model. API will run in mock mode.")

@app.post("/predict", response_model=PredictionResponse)
async def predict_anxiety(request: TextRequest):
    """Endpoint to predict anxiety level from input text."""
    if not request.text.strip():
        raise HTTPException(status_code=400, detail="Text input cannot be empty.")

    # Fallback/Mock mode if model isn't trained yet (prevents backend crashes during UI testing)
    if model is None or tokenizer is None:
        text_lower = request.text.lower()
        mock_label = 1
        if "panic" in text_lower or "terrified" in text_lower or "cry" in text_lower:
            mock_label = 2
        elif "calm" in text_lower or "ready" in text_lower or "good" in text_lower:
            mock_label = 0

        return PredictionResponse(
            anxiety_level=LABEL_MAP[mock_label],
            confidence=0.99
        )

    # Actual AI Inference
    try:
        # Tokenize input text
        inputs = tokenizer(request.text, return_tensors="pt", truncation=True, padding=True, max_length=128)

        # Predict
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=-1)
            predicted_class_id = torch.argmax(probabilities, dim=-1).item()
            confidence = probabilities[0][predicted_class_id].item()

        return PredictionResponse(
            anxiety_level=LABEL_MAP[predicted_class_id],
            confidence=round(confidence, 4)
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
async def health_check():
    return {"status": "API is running. Use /predict for inference."}